# Test mécanique du fine-tune Omnilingual — baoulé (bci)

Protocole : `docs/benchmarks/0005-omnilingual-finetune-smoke-test.md` — Wourri, issue #474 (étape 2 de l'étude 0004).

**But** : prouver que la **recette officielle de fine-tune** d'Omnilingual tourne **de bout en bout** sur nos
données, sur un GPU gratuit, **avant** le vrai run. Ce n'est **pas** la mesure du CER final : 100 étapes ne
suffisent pas à entraîner un modèle.

Deux essais de 100 étapes, sur Common Voice `bci` (+ la dictée d'esse en option) — le split *test* du
benchmark 0003 reste hors entraînement :
- **A** — `omniASR_CTC_300M`, fine-tune **complet** (voie A de l'étude 0004) ;
- **B** — `omniASR_CTC_1B`, **encodeur gelé** pendant tout l'essai (voie B).

**Runtime** : Kaggle → *Settings → Accelerator → GPU T4 x2* + *Internet on*. Un seul GPU est utilisé.
Le Python du noyau Kaggle n'a pas d'importance : fairseq2 0.6 n'existe que jusqu'à Python 3.12, le notebook
crée donc son **propre Python 3.12** (avec `uv`) et y lance préparation des données et entraînement.
**Aucun redémarrage de session. Aucun token Hugging Face requis.**

> ⚠️ Exécuter les cellules **dans l'ordre**. Si le noyau redémarre, ré-exécuter la cellule 1 puis reprendre.

In [ ]:
# Cellule 1 — GPU, chemins et lanceur commun (a re-executer si le noyau redemarre)
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path.cwd()                      # /kaggle/working (ou /content sur Colab)
VENV = Path("/tmp/omni312")            # Python 3.12 dedie : fairseq2 0.6 n'existe pas en 3.13
VENV_PY = str(VENV / "bin" / "python")

def sh(args, cwd=None, env=None, log_path=None, check=True):
    """Lance une commande en affichant sa sortie en direct (copiee dans log_path) ; retourne le code."""
    proc = subprocess.Popen([str(a) for a in args], cwd=cwd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    log = open(log_path, "w", encoding="utf-8") if log_path else None
    for line in proc.stdout:
        print(line, end="")
        if log:
            log.write(line)
    if log:
        log.close()
    code = proc.wait()
    if check and code != 0:
        raise SystemExit(f"ECHEC (code {code}) : {' '.join(str(a) for a in args)[:200]}")
    return code

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout
      or "PAS DE GPU -> Kaggle: Settings > Accelerator > GPU T4 x2")
print("Python du noyau :", sys.version.split()[0], "(sans importance : les essais tournent dans", VENV, ")")

## Étape 1 — Installation (Python 3.12 dédié)

Mêmes versions que le benchmark 0003 : `omnilingual-asr==0.1.0`, `fairseq2[arrow]==0.6`, `torch==2.8.0`
(fairseq2n 0.6 exige exactement cette version de torch et n'a de roues que pour Python 3.10 à 3.12).
Le dépôt Omnilingual est cloné au **même tag** que le paquet : la recette (`workflows/`) n'est pas livrée par
`pip`. Compter quelques minutes (torch + bibliothèques CUDA).

In [ ]:
# Cellule 2 — Python 3.12 (uv) + versions figees du benchmark 0003 + recette au meme tag
# librosa<1.0 : le webm de la dictee se decode via audioread + ffmpeg, retire en 1.0.
UV = [sys.executable, "-m", "uv"]
sh([sys.executable, "-m", "pip", "install", "--quiet", "uv", "pyyaml"])
sh(UV + ["venv", "--quiet", "--python", "3.12", VENV])
sh(UV + ["pip", "install", "--quiet", "--python", VENV_PY,
         "omnilingual-asr==0.1.0", "fairseq2[arrow]==0.6", "torch==2.8.0", "torchaudio==2.8.0",
         "datasets<4.0", "soundfile", "librosa<1.0", "pyyaml"])
shutil.rmtree("omnilingual-asr", ignore_errors=True)
sh(["git", "clone", "--quiet", "--depth", "1", "--branch", "0.1.0",
    "https://github.com/facebookresearch/omnilingual-asr.git"])
sh([VENV_PY, "-c", "import sys, torch, fairseq2, omnilingual_asr; "
    "print('Python', sys.version.split()[0], '| torch', torch.__version__, '| GPU', torch.cuda.is_available())"])
print("\n>>> INSTALLATION TERMINEE (pas de redemarrage necessaire) -> cellule 3.")

## Étape 2 — Données

La cellule 3 écrit le convertisseur du dépôt (`finetune/dictee_to_parquet.py`, copie identique vérifiée par un
test) ; la cellule 4 écrit le script de préparation ; la cellule 5 l'exécute dans le Python 3.12 et produit le
dataset **partitionné** `corpus=/split=/language=` que lit la recette, ses statistiques et sa carte fairseq2.

In [ ]:
%%writefile dictee_to_parquet.py
"""Convertit l'export dictée (ZIP audiofolder) en dataset parquet Omnilingual — maillon ③ (#474).

Pont entre le maillon ② (export dictée, ADR-0035) et le maillon ③ (fine-tune
Omnilingual, cf. étude de faisabilité docs/benchmarks/0004 et test mécanique 0005).

Entrée : ZIP export dictée
    audio/<x>.webm ...
    metadata.csv : file_name, transcription, language, filiere, text_fr

Sortie : un dataset parquet partitionné (« hive »), seul format que lit le chargeur
`MixtureParquetStorage` de la recette de fine-tune (omnilingual-asr 0.1.0) :
    <racine>/corpus=<corpus>/split=<split>/language=<code>/part-0.parquet
        - text        : transcription normalisée (baoulé) — la CIBLE ASR
        - audio_bytes : audio FLAC 16 kHz mono, en list<int8> (schéma amont)
        - audio_size  : nombre d'échantillons du waveform décodé
    corpus / split / language vivent dans les NOMS DE DOSSIERS, pas dans le fichier :
    la recette découvre les splits et les partitions à partir de l'arborescence.
    Un fichier parquet « à plat » n'est pas lu (aucun split découvert).

Le code langue est celui d'Omnilingual (ISO 639-3 + écriture) : `bci` -> `bci_Latn`.

Usage :
    python dictee_to_parquet.py export.zip dataset_root \\
        --corpus wourri_dictee --split train --language bci \\
        --stats language_distribution_0.tsv

Décodage : soundfile (wav/flac/ogg) ; repli librosa+ffmpeg pour webm/opus/mp3
(le fine-tune tourne sur Colab/Kaggle où ffmpeg est présent). Le rééchantillonnage
utilise librosa si présent, sinon un repli linéaire (dégradé — le vrai run a librosa).

Ce module est recopié tel quel dans le notebook `colab/omnilingual_finetune_smoke_test.ipynb`
(cellule %%writefile) : un test vérifie que les deux copies restent identiques.
"""
from __future__ import annotations

import argparse
import csv
import io
import re
import sys
import unicodedata
import zipfile
from pathlib import Path, PurePosixPath

TARGET_SR = 16000
ROW_GROUP_SIZE = 100  # valeur amont : limite la mémoire du streaming et permet le mélange
FILE_COLUMNS = ["text", "audio_bytes", "audio_size"]

# Apostrophe baoulé : Common Voice bci et la sortie d'Omnilingual l'écrivent U+02BC (ʼ,
# lettre modificative, conservée par la normalisation amont et présente dans le vocabulaire :
# le 1B la produit au benchmark 0003). Les variantes tapées au clavier ENTRE DEUX LETTRES
# (' ’ ‘ ‛) y sont ramenées, sinon un même son aurait plusieurs cibles ; ailleurs ce sont
# des guillemets, retirés comme la ponctuation. Un ton combinant (ɛ̀ = ɛ + U+0300, sans forme
# précomposée) compte comme lettre.
_APOSTROPHES = "'‘’‛"
_INNER_APOSTROPHE = re.compile(rf"(?<=[\ẁ-ͯ])[{_APOSTROPHES}](?=\w)")
_OTHER_APOSTROPHE = re.compile(rf"[{_APOSTROPHES}]")
# Invisibles supprimés par l'amont (shared_deletion_list) + BOM : souvent hérités d'un copier-coller.
_INVISIBLE = re.compile("[​‌‎‏‪‬﻿]")
_PUNCT = re.compile(r"[.,!?;:«»\"“”„()\[\]{}…—–/\\_]")
_DIGIT_WORD = re.compile(r"(?<!\S)\d+(?!\S)")


def normalize_text(text: str) -> str:
    """Normalise une transcription comme la préparation de données amont (text_tools) :
    NFKC, invisibles supprimés, minuscules, ponctuation retirée, mots uniquement numériques
    retirés. Les lettres baoulé (ɛ ɔ ɲ, tons, ʼ) sont conservées : elles portent le sens."""
    t = unicodedata.normalize("NFKC", text or "")
    t = _INVISIBLE.sub("", t).lower()
    t = _INNER_APOSTROPHE.sub("ʼ", t)
    t = _OTHER_APOSTROPHE.sub(" ", t)
    t = _PUNCT.sub(" ", t)
    t = _DIGIT_WORD.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()


def omni_language_code(code: str) -> str:
    """Code Omnilingual (`lang_ids.py`) = ISO 639-3 + écriture : 'bci' -> 'bci_Latn'.

    Les langues de l'atelier s'écrivent en latin (colonne `script` de la table
    `languages`) ; un code déjà complet ('bci_Latn') est gardé tel quel."""
    code = (code or "").strip()
    if not code:
        raise ValueError("code langue vide")
    return code if "_" in code else f"{code}_Latn"


def decode_to_16k_mono(raw: bytes, filename: str):
    """Octets audio -> (waveform float32 mono 16 kHz, n_samples)."""
    import numpy as np
    import soundfile as sf

    try:
        wav, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)  # (n, ch)
        wav = wav.mean(axis=1)  # -> mono (n,)
    except Exception:
        wav, sr = _decode_via_librosa(raw, filename)  # webm/opus/mp3 -> ffmpeg
    if sr != TARGET_SR:
        wav = _resample(np.asarray(wav, dtype="float32"), sr, TARGET_SR)
    return np.asarray(wav, dtype="float32"), int(len(wav))


def _decode_via_librosa(raw: bytes, filename: str):
    import tempfile

    import librosa

    suffix = Path(filename).suffix or ".webm"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(raw)
        path = tmp.name
    try:
        wav, sr = librosa.load(path, sr=None, mono=True)  # float32 mono
    finally:
        Path(path).unlink(missing_ok=True)
    return wav, sr


def _resample(wav, sr: int, target: int):
    import numpy as np

    try:
        import librosa

        return librosa.resample(wav, orig_sr=sr, target_sr=target)
    except Exception:
        # repli linéaire (le vrai run a librosa ; ce repli garde le script utilisable en test)
        n = int(round(len(wav) * target / sr))
        if n <= 0:
            return wav
        return np.interp(
            np.linspace(0, len(wav) - 1, n), np.arange(len(wav)), wav
        ).astype("float32")


def encode_flac(wav) -> bytes:
    """Waveform float32 mono 16 kHz -> octets FLAC."""
    import soundfile as sf

    buf = io.BytesIO()
    sf.write(buf, wav, TARGET_SR, format="FLAC")
    return buf.getvalue()


def read_export(zip_path) -> list[dict]:
    """Lit le ZIP export dictée -> [{file_name, transcription, language, filiere, text_fr, raw}]."""
    rows: list[dict] = []
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        metas = sorted(n for n in names if PurePosixPath(n).name == "metadata.csv")
        if not metas:
            raise SystemExit("metadata.csv introuvable dans le ZIP")
        meta_name = "metadata.csv" if "metadata.csv" in names else metas[0]
        meta = zf.read(meta_name).decode("utf-8-sig")
        for r in csv.DictReader(io.StringIO(meta)):
            fn = (r.get("file_name") or "").strip()
            if not fn:
                continue
            if fn not in names:
                # tolère un préfixe de chemin différent, jamais un autre fichier :
                # même nom exact (pas « 11.webm » pour « 1.webm ») et un seul candidat
                base = PurePosixPath(fn).name
                cands = [n for n in names if PurePosixPath(n).name == base]
                if len(cands) != 1:
                    why = "absent du ZIP" if not cands else f"ambigu ({len(cands)} fichiers)"
                    print(f"  [SKIP] audio {why} : {fn}", file=sys.stderr)
                    continue
                fn = cands[0]
            rows.append(
                {
                    "file_name": fn,
                    "transcription": (r.get("transcription") or "").strip(),
                    "language": (r.get("language") or "").strip(),
                    "filiere": (r.get("filiere") or "").strip(),
                    "text_fr": (r.get("text_fr") or "").strip(),
                    "raw": zf.read(fn),
                }
            )
    return rows


def make_table(examples):
    """[(transcription, waveform float32 mono 16 kHz)] -> table parquet (colonnes FILE_COLUMNS).

    Ignore les paires sans texte après normalisation ou sans audio : une paire
    audio↔texte complète est requise pour l'apprentissage."""
    import numpy as np
    import pyarrow as pa

    texts, audios, sizes = [], [], []
    for text, wav in examples:
        norm = normalize_text(text)
        if not norm or len(wav) == 0:
            continue
        texts.append(norm)
        audios.append(np.frombuffer(encode_flac(wav), dtype=np.int8))
        sizes.append(int(len(wav)))
    return pa.table(
        {
            "text": pa.array(texts, type=pa.string()),
            "audio_bytes": pa.array(audios, type=pa.list_(pa.int8())),
            "audio_size": pa.array(sizes, type=pa.int64()),
        }
    )


def write_partition(table, root, corpus: str, split: str, language: str) -> Path:
    """Écrit la table dans <root>/corpus=…/split=…/language=…/part-0.parquet.

    Remplace les parquet déjà présents dans CETTE partition : un export dictée est
    toujours complet, le reconvertir doit remplacer l'ancien, pas s'y ajouter."""
    import pyarrow.parquet as pq

    for key, value in (("corpus", corpus), ("split", split), ("language", language)):
        if not value or "/" in value or "=" in value:
            raise ValueError(f"{key} invalide pour un nom de partition : {value!r}")
    if "_" in split:
        # la recette lit un split « <split>_<corpus> » comme un filtre de corpus
        raise ValueError(f"split sans '_' attendu (reçu {split!r})")
    if table.num_rows == 0:
        # une partition vide fait échouer la recette au démarrage (lecture de son 1er exemple)
        raise ValueError(f"aucune paire audio↔texte exploitable pour {corpus}/{split}")
    part_dir = Path(root) / f"corpus={corpus}" / f"split={split}" / f"language={language}"
    part_dir.mkdir(parents=True, exist_ok=True)
    for old in part_dir.glob("*.parquet"):
        old.unlink()
    out = part_dir / "part-0.parquet"
    pq.write_table(table, out, row_group_size=ROW_GROUP_SIZE)
    return out


def write_stats(root, out_tsv) -> list[dict]:
    """Écrit le TSV `corpus / language / hours` attendu par `dataset_summary_path`
    (pondération des partitions à l'entraînement). Comme l'outil amont, les heures
    sont cumulées sur tous les splits."""
    import pyarrow.dataset as ds

    table = ds.dataset(str(root), format="parquet", partitioning="hive").to_table(
        columns=["corpus", "language", "audio_size"]
    )
    samples: dict[tuple[str, str], int] = {}
    for corpus, language, size in zip(
        table.column("corpus").to_pylist(),
        table.column("language").to_pylist(),
        table.column("audio_size").to_pylist(),
    ):
        key = (str(corpus), str(language))
        samples[key] = samples.get(key, 0) + int(size)
    stats = [
        {"corpus": c, "language": lang, "hours": n / TARGET_SR / 3600}
        for (c, lang), n in sorted(samples.items())
    ]
    lines = ["corpus\tlanguage\thours"]
    lines += [f"{s['corpus']}\t{s['language']}\t{s['hours']:.6f}" for s in stats]
    Path(out_tsv).parent.mkdir(parents=True, exist_ok=True)
    Path(out_tsv).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return stats


def build_dataset(rows, root, corpus: str, split: str, language: str) -> int:
    """Lignes d'export dictée -> partition parquet. Retourne le nombre de clips écrits."""

    def examples():
        for r in rows:
            if r.get("transcription"):
                wav, _ = decode_to_16k_mono(r["raw"], r["file_name"])
                yield r["transcription"], wav

    table = make_table(examples())
    write_partition(table, root, corpus, split, omni_language_code(language))
    return table.num_rows


def main():
    ap = argparse.ArgumentParser(
        description="Export dictée (ZIP audiofolder) -> dataset parquet Omnilingual (#474)"
    )
    ap.add_argument("zip_path", help="ZIP export dictée (audio/ + metadata.csv)")
    ap.add_argument("out_root", help="racine du dataset (le `data:` de la carte fairseq2)")
    ap.add_argument("--corpus", default="wourri_dictee")
    ap.add_argument("--split", default="train", choices=["train", "dev", "test"])
    ap.add_argument("--language", default=None, help="code langue (ex. bci) ; défaut : celui du CSV")
    ap.add_argument("--stats", default=None, help="écrit aussi le TSV corpus/language/hours")
    args = ap.parse_args()

    rows = read_export(args.zip_path)
    language = args.language
    if not language:
        found = {r["language"] for r in rows if r.get("language")}
        if len(found) != 1:
            raise SystemExit(f"--language requis : langues trouvées dans le CSV = {sorted(found)}")
        language = found.pop()
    n = build_dataset(rows, args.out_root, args.corpus, args.split, language)
    print(f"OK : {n} clips -> {args.out_root} (corpus={args.corpus}, split={args.split}, "
          f"language={omni_language_code(language)})")
    if args.stats:
        write_stats(args.out_root, args.stats)
        print(f"Stats -> {args.stats}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile prepare_data.py
"""Prepare le dataset du test 0005 — execute dans le Python 3.12 dedie (cellule 5).

Common Voice bci (train -> train, validation -> dev) + export dictee optionnel (train), puis le
TSV de statistiques et la carte dataset fairseq2. Le split "test" (290 clips) reste HORS
entrainement : c'est le set du benchmark 0003 (CER zero-shot 1B : 22,1 % ; 300M : 26,0 %).
"""
import argparse
import shutil
from pathlib import Path

import numpy as np

import dictee_to_parquet as d2p

REV = "176d5f8ad6c04e5ca1a0e66770866709bcbb338b"   # meme revision que le benchmark 0003


def get_audio(row):
    a = row["audio"]
    if isinstance(a, dict):                        # datasets < 4.0
        return np.asarray(a["array"], dtype="float32"), int(a["sampling_rate"])
    samples = a.get_all_samples()                  # datasets >= 4.0 (AudioDecoder)
    return samples.data.numpy().squeeze().astype("float32"), int(samples.sample_rate)


def cv_examples(hf_split):
    from datasets import load_dataset
    for row in load_dataset("Klayt/baoule-common-voice", revision=REV, split=hf_split):
        wav, sr = get_audio(row)
        if sr != d2p.TARGET_SR:
            import librosa
            wav = librosa.resample(wav, orig_sr=sr, target_sr=d2p.TARGET_SR)
        yield row["sentence"], wav


def find_export(path):
    """Le ZIP de l'ecran Dictee, ou son dossier decompresse par Kaggle (rezippe au format attendu)."""
    src = Path(path)
    if src.is_file():
        return src
    zips = sorted(src.rglob("dictee_*.zip"))
    if zips:
        return zips[0]
    metas = sorted(src.rglob("metadata.csv"))
    if metas:
        return Path(shutil.make_archive("/tmp/dictee_export", "zip", root_dir=metas[0].parent))
    raise SystemExit(f"Ni dictee_*.zip ni metadata.csv sous {path}")


def write_card(asset_dir, data_root):
    asset_dir.mkdir(parents=True, exist_ok=True)
    card = asset_dir / "wourri_bci.yaml"
    card.write_text(
        "name: wourri_bci\n"
        "dataset_family: mixture_parquet_asr_dataset\n"
        "dataset_config:\n"
        f"  data: {data_root}\n"
        "tokenizer_ref: omniASR_tokenizer\n",
        encoding="utf-8",
    )
    return card


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--work", required=True)
    ap.add_argument("--dictee", default=None)
    args = ap.parse_args()
    work = Path(args.work)
    dataset_dir = work / "wourri_bci"
    data_root = dataset_dir / "version=0"          # le `data:` de la carte fairseq2
    shutil.rmtree(dataset_dir, ignore_errors=True) # repartir propre a chaque preparation
    lang = d2p.omni_language_code("bci")           # bci_Latn

    for hf_split, split in (("train", "train"), ("validation", "dev")):
        table = d2p.make_table(cv_examples(hf_split))
        d2p.write_partition(table, data_root, "cv_bci", split, lang)
        print(f"cv_bci/{split}: {table.num_rows} clips")
    print("Exemple de cible normalisee :", table.column("text")[0].as_py())

    if args.dictee:
        export = find_export(args.dictee)
        rows = d2p.read_export(export)
        n = d2p.build_dataset(rows, data_root, "wourri_dictee", "train", "bci")
        print(f"wourri_dictee/train: {n} clips (sur {len(rows)} lignes de {export.name})")
    else:
        print("Pas d'export dictee : Common Voice bci seul.")

    for s in d2p.write_stats(data_root, dataset_dir / "language_distribution_0.tsv"):
        print(f"{s['corpus']:15s} {s['language']}  {s['hours']:.2f} h")
    print("Carte :", write_card(work / "fairseq2_assets", data_root))


if __name__ == "__main__":
    main()

In [ ]:
# Cellule 5 — preparer les donnees dans le Python 3.12
# DICTEE : l'export de l'ecran Dictee depose en dataset Kaggle PRIVE. Mettre "/kaggle/input"
# (le ZIP ou son dossier decompresse y est retrouve). Laisser None pour Common Voice seul.
DICTEE = None
sh([VENV_PY, "prepare_data.py", "--work", WORK] + (["--dictee", DICTEE] if DICTEE else []))
STATS_TSV = WORK / "wourri_bci" / "language_distribution_0.tsv"
# Repertoire d'assets utilisateur de fairseq2 : la recette y trouve la carte `wourri_bci` par son nom.
os.environ["FAIRSEQ2_USER_ASSET_DIR"] = str(WORK / "fairseq2_assets")

In [ ]:
# Cellule 6 — configs des deux essais, derivees de la config "recommandation" amont.
# --dump-config fait fusionner la config par fairseq2 (cles et types) sans entrainer ;
# la carte dataset, elle, n'est ouverte qu'au lancement de l'essai.
import copy, yaml

NUM_STEPS = 100
MAX_NUM_ELEMENTS = 960_000   # 60 s d'audio par lot : vise un T4 16 Go (amont : 7,68 M sur gros GPU)
RECO = "omnilingual-asr/workflows/recipes/wav2vec2/asr/configs/ctc-finetune-recommendation.yaml"
with open(RECO, encoding="utf-8") as f:
    BASE = yaml.safe_load(f)

def smoke_config(model_name, freeze_encoder_steps):
    c = copy.deepcopy(BASE)
    c["model"]["name"] = model_name
    c["dataset"]["name"] = "wourri_bci"
    c["dataset"]["mixture_parquet_storage_config"]["dataset_summary_path"] = str(STATS_TSV)
    c["dataset"]["asr_task_config"]["max_num_elements"] = MAX_NUM_ELEMENTS
    c["trainer"]["freeze_encoder_for_n_steps"] = freeze_encoder_steps
    # Le T4 (Turing) n'a pas de bfloat16 materiel : tout en float32 (mode "off").
    c["trainer"]["mixed_precision"] = {"mode": "off", "dtype": "torch.float32"}
    c["trainer"]["grad_accumulation"]["num_batches"] = 4
    c["regime"] = {
        "num_steps": NUM_STEPS,
        "validate_at_start": True,          # UER/WER AVANT entrainement = point de depart
        "validate_after_n_steps": 0,
        "validate_every_n_steps": NUM_STEPS // 2,
        "checkpoint_after_n_steps": 0,
        "checkpoint_every_n_steps": NUM_STEPS,
        "publish_metrics_after_n_steps": 0,
        "publish_metrics_every_n_steps": 10,
        "save_model_only": True,
        "keep_last_n_checkpoints": 1,
    }
    return c

CONFIGS = {
    "A_300m_complet": smoke_config("omniASR_CTC_300M", 0),
    "B_1b_encodeur_gele": smoke_config("omniASR_CTC_1B", NUM_STEPS),  # gele pendant tout l'essai
}
for tag, cfg in CONFIGS.items():
    path = WORK / f"smoke_{tag}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
    r = subprocess.run(
        [VENV_PY, "-m", "workflows.recipes.wav2vec2.asr", "--config-file", str(path), "--dump-config"],
        cwd="omnilingual-asr", capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit(f"{tag} : config refusee par fairseq2")
    print(f"{tag} : config acceptee par fairseq2 -> {path}")

In [ ]:
# Cellule 7 — lanceur des essais : journal complet + pic memoire GPU (releve toutes les 2 s)
import json, re, threading, time

RESULTS_PATH = WORK / "finetune_smoke_results.json"

def gpu_memory_mib():
    out = subprocess.run(
        ["nvidia-smi", "-i", "0", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True).stdout.strip()
    return int(out) if out.isdigit() else None

def run(tag):
    out_dir = WORK / f"run_{tag}"
    shutil.rmtree(out_dir, ignore_errors=True)   # sinon fairseq2 reprend au dernier checkpoint
    log_path = WORK / f"run_{tag}.log"
    # Un seul GPU (Kaggle en fournit 2). PCI_BUS_ID : CUDA numerote alors les GPU comme
    # nvidia-smi, le GPU 0 entraine est bien celui dont on releve la memoire.
    env = {**os.environ, "CUDA_DEVICE_ORDER": "PCI_BUS_ID", "CUDA_VISIBLE_DEVICES": "0"}
    peak, done = [0], threading.Event()
    def watch():   # releves ponctuels : rien ne reste dans un tampon a l'arret
        while not done.wait(2):
            used = gpu_memory_mib()
            if used:
                peak[0] = max(peak[0], used)
    watcher = threading.Thread(target=watch, daemon=True)
    watcher.start()
    t0 = time.time()
    code = sh([VENV_PY, "-m", "workflows.recipes.wav2vec2.asr", out_dir,
               "--config-file", WORK / f"smoke_{tag}.yaml"],
              cwd="omnilingual-asr", env=env, log_path=log_path, check=False)
    done.set()
    watcher.join()
    lines = log_path.read_text(encoding="utf-8").splitlines()
    ckpt_dir = out_dir / "checkpoints"
    result = {
        "exit_code": code,
        "minutes": round((time.time() - t0) / 60, 1),
        "peak_gpu_mib": peak[0] or None,
        "checkpoints": sorted(p.name for p in ckpt_dir.glob("step_*")) if ckpt_dir.exists() else [],
        "oom": any("out of memory" in l.lower() for l in lines),
        "metric_lines": [l for l in lines if re.search(r"\b(uer|wer)\b", l, re.I)][-8:],
    }
    results = json.loads(RESULTS_PATH.read_text(encoding="utf-8")) if RESULTS_PATH.exists() else {}
    results[tag] = result
    RESULTS_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n>>> {tag} : code {code}, {result['minutes']} min, pic GPU {result['peak_gpu_mib']} Mio, "
          f"checkpoints {result['checkpoints']}")
    return result

## Étape 3 — Les deux essais

Chaque essai affiche la validation **au départ** (modèle non fine-tuné), à l'étape 50 et à l'étape 100.
Si l'essai A échoue, l'essai B reste utile : lance-le quand même.

In [ ]:
# Cellule 8 — essai A : 300M, fine-tune complet (voie A de l'etude 0004)
run("A_300m_complet");

In [ ]:
# Cellule 9 — essai B : 1B, encodeur gele pendant les 100 etapes (voie B)
run("B_1b_encodeur_gele");

In [ ]:
# Cellule 10 — synthese a reporter dans docs/benchmarks/0005 (section Resultats)
results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
for tag, r in results.items():
    ok = r["exit_code"] == 0 and f"step_{NUM_STEPS}" in r["checkpoints"] and not r["oom"]
    print(f"\n=== {tag} : {'REUSSI - la recette tourne de bout en bout' if ok else 'ECHEC'}")
    print(f"code {r['exit_code']} | {r['minutes']} min | pic GPU {r['peak_gpu_mib']} Mio | OOM {r['oom']}")
    for l in r["metric_lines"]:
        print("   ", l.strip()[:220])
print("\nResultats complets ->", RESULTS_PATH, "(+ journaux run_*.log)")

## Lecture

Un essai est **réussi** si : code de sortie 0, checkpoint `step_100` écrit, pas d'OOM, et validation affichée au
départ puis à l'étape 100 (critères du protocole 0005 §4). Une baisse de la perte ou de l'UER est un bon signe,
mais **pas** un critère : 100 étapes, c'est trop peu pour conclure sur la qualité.

1. **Reporte** la synthèse (cellule 10) dans `docs/benchmarks/0005-omnilingual-finetune-smoke-test.md`.
2. Ces chiffres (temps, pic mémoire, voies qui tiennent sur T4) alimentent l'**ADR « procédure de fine-tune »**
   (choix A / B / C de l'étude 0004), puis le vrai run sur la dictée d'esse.